# 🌧️ RainVision AI — Complete Google Colab Notebook

**AI-Based Rainfall Prediction Using LSTM Deep Learning**  
Location: Hyderabad, Telangana, India  
Historical data period: 2015-01-01 to 2025-12-31

## How to run
1. Open this notebook in Google Colab.
2. Choose **Runtime → Change runtime type → T4 GPU** if available (CPU also works).
3. Choose **Runtime → Run all** and wait for the cells to finish.
4. The notebook downloads historical weather, trains an LSTM, evaluates it, builds a Gradio dashboard, and saves the project artifacts.

**Research limitations:** The Open-Meteo historical archive provides gridded/reanalysis weather estimates, not necessarily local rain-gauge observations. The seven-day forecast is recursive and holds future temperature, humidity, pressure, and wind speed at their last observed values because the model predicts rainfall only. It is an academic experiment, not a verified operational weather forecast.

## 1. Install dependencies

In [14]:
%pip install -q tensorflow gradio plotly scikit-learn joblib

## 2. Imports and reproducibility

In [2]:
import os, random, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import gradio as gr
import joblib
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Libraries imported successfully")
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

Libraries imported successfully
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Download historical Hyderabad weather data

In [3]:
LATITUDE, LONGITUDE = 17.3850, 78.4867
START_DATE, END_DATE = "2015-01-01", "2025-12-31"

api_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "daily": [
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "pressure_msl_mean",
        "wind_speed_10m_max",
        "precipitation_sum"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(api_url, params=params, timeout=120)
response.raise_for_status()
daily = response.json().get("daily", {})

if not daily or not daily.get("time"):
    raise RuntimeError("The weather API returned no daily data. Please rerun this cell later.")

df = pd.DataFrame({
    "Date": pd.to_datetime(daily["time"]),
    "Temperature": daily["temperature_2m_mean"],
    "Humidity": daily["relative_humidity_2m_mean"],
    "Pressure": daily["pressure_msl_mean"],
    "WindSpeed": daily["wind_speed_10m_max"],
    "Rainfall": daily["precipitation_sum"]
}).dropna().sort_values("Date").reset_index(drop=True)

df.to_csv("hyderabad_rainfall_dataset.csv", index=False)
print("Dataset downloaded:", df.shape)
display(df.head())
display(df.tail())

Dataset downloaded: (4018, 6)


,Date,Temperature,Humidity,Pressure,WindSpeed,Rainfall
0,2015-01-01,23.3,69,1009.8,15.7,0.2
1,2015-01-02,23.1,72,1012.4,12.8,0.0
2,2015-01-03,22.4,84,1014.1,10.3,4.0
3,2015-01-04,21.7,85,1015.1,9.3,0.6
4,2015-01-05,22.1,70,1014.9,13.6,0.0


,Date,Temperature,Humidity,Pressure,WindSpeed,Rainfall
4013,2025-12-27,21.5,53,1015.3,8.0,0.0
4014,2025-12-28,21.5,52,1015.8,8.0,0.0
4015,2025-12-29,21.4,51,1015.5,8.0,0.0
4016,2025-12-30,22.0,52,1014.3,5.1,0.0
4017,2025-12-31,22.7,53,1012.4,9.4,0.0


## 4. Explore historical rainfall

In [4]:
display(df["Rainfall"].describe())

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["Date"], y=df["Rainfall"], mode="lines", name="Daily rainfall"))
fig.update_layout(
    title="Historical Daily Rainfall — Hyderabad",
    xaxis_title="Date", yaxis_title="Rainfall (mm)",
    template="plotly_white", height=420
)
fig.show()

,Rainfall
count,4018.000000
mean,2.420159
std,6.045853
min,0.000000
25%,0.000000
50%,0.000000
75%,1.700000
max,82.700000


## 5. Scale features and create 7-day sequences

In [5]:
features = ["Temperature", "Humidity", "Pressure", "WindSpeed", "Rainfall"]
data = df[features].astype(float).values

if len(data) < 100:
    raise ValueError("The downloaded dataset is unexpectedly small.")

split_index = int(len(data) * 0.8)

# Fit scalers only on the chronological training period to avoid test leakage.
feature_scaler = MinMaxScaler().fit(data[:split_index])
target_scaler = MinMaxScaler().fit(data[:split_index, 4].reshape(-1, 1))

X_scaled = feature_scaler.transform(data)
y_scaled = target_scaler.transform(data[:, 4].reshape(-1, 1))

sequence_length = 7
X, y = [], []
for i in range(sequence_length, len(X_scaled)):
    X.append(X_scaled[i-7:i])
    y.append(y_scaled[i, 0])

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.float32)

target_indices = np.arange(sequence_length, len(X_scaled))
train_mask = target_indices < split_index
test_mask = target_indices >= split_index

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print("All sequences:", X.shape)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

All sequences: (4011, 7, 5)
Training samples: 3207
Testing samples: 804


## 6. Build and train the LSTM

In [6]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(7, 5)),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])

early_stop = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    shuffle=False,
    verbose=1
)

print("LSTM training completed.")

Epoch 1/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 12s 46ms/step - loss: 0.0044 - mae: 0.0351 - val_loss: 0.0064 - val_mae: 0.0347
Epoch 2/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0041 - mae: 0.0310 - val_loss: 0.0063 - val_mae: 0.0340
Epoch 3/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0041 - mae: 0.0310 - val_loss: 0.0062 - val_mae: 0.0349
Epoch 4/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0039 - mae: 0.0303 - val_loss: 0.0064 - val_mae: 0.0348
Epoch 5/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0040 - mae: 0.0299 - val_loss: 0.0060 - val_mae: 0.0345
Epoch 6/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0040 - mae: 0.0314 - val_loss: 0.0060 - val_mae: 0.0346
Epoch 7/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0038 - mae: 0.0291 - val_loss: 0.0059 - val_mae: 0.0336
Epoch 8/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0038 - mae: 0.0291 - val_loss: 0.0059 - val_mae: 0.0335
Epoch 9/30
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0037 - mae

## 7. Training and validation loss

In [7]:
loss_fig = go.Figure()
loss_fig.add_trace(go.Scatter(y=history.history["loss"], mode="lines+markers", name="Training loss"))
loss_fig.add_trace(go.Scatter(y=history.history["val_loss"], mode="lines+markers", name="Validation loss"))
loss_fig.update_layout(title="LSTM Training and Validation Loss", xaxis_title="Epoch",
                       yaxis_title="MSE loss", template="plotly_white", height=400)
loss_fig.show()

## 8. One-day test evaluation

In [8]:
pred_scaled = model.predict(X_test, verbose=0)
predicted_rainfall = np.maximum(
    target_scaler.inverse_transform(pred_scaled).flatten(), 0
)
actual_rainfall = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

one_day_mae = mean_absolute_error(actual_rainfall, predicted_rainfall)
one_day_rmse = np.sqrt(mean_squared_error(actual_rainfall, predicted_rainfall))

zero_baseline = np.zeros_like(actual_rainfall)
training_mean_baseline = np.full_like(actual_rainfall, data[:split_index, 4].mean())

zero_mae = mean_absolute_error(actual_rainfall, zero_baseline)
zero_rmse = np.sqrt(mean_squared_error(actual_rainfall, zero_baseline))
mean_mae = mean_absolute_error(actual_rainfall, training_mean_baseline)
mean_rmse = np.sqrt(mean_squared_error(actual_rainfall, training_mean_baseline))

print("ONE-DAY TEST EVALUATION")
print(f"LSTM MAE: {one_day_mae:.3f} mm")
print(f"LSTM RMSE: {one_day_rmse:.3f} mm")
print(f"Zero baseline MAE: {zero_mae:.3f} mm")
print(f"Zero baseline RMSE: {zero_rmse:.3f} mm")
print(f"Training mean baseline MAE: {mean_mae:.3f} mm")
print(f"Training mean baseline RMSE: {mean_rmse:.3f} mm")

fig = go.Figure()
fig.add_trace(go.Scatter(y=actual_rainfall[:150], mode="lines", name="Actual"))
fig.add_trace(go.Scatter(y=predicted_rainfall[:150], mode="lines", name="LSTM prediction"))
fig.update_layout(title="One-Day Test: Actual vs Predicted Rainfall",
                  xaxis_title="Test observation", yaxis_title="Rainfall (mm)",
                  template="plotly_white", height=420)
fig.show()

ONE-DAY TEST EVALUATION
LSTM MAE: 2.179 mm
LSTM RMSE: 5.045 mm
Zero baseline MAE: 2.737 mm
Zero baseline RMSE: 6.616 mm
Training mean baseline MAE: 3.516 mm
Training mean baseline RMSE: 6.036 mm


## 9. Threshold-based rain/no-rain metrics

In [9]:
# Regression outputs are thresholded for this secondary classification-style analysis.
RAIN_THRESHOLD_MM = 0.1
actual_classes = (actual_rainfall >= RAIN_THRESHOLD_MM).astype(int)
predicted_classes = (predicted_rainfall >= RAIN_THRESHOLD_MM).astype(int)

dashboard_accuracy = accuracy_score(actual_classes, predicted_classes)
dashboard_precision = precision_score(actual_classes, predicted_classes, zero_division=0)
dashboard_recall = recall_score(actual_classes, predicted_classes, zero_division=0)
dashboard_f1 = f1_score(actual_classes, predicted_classes, zero_division=0)

print(f"Threshold: {RAIN_THRESHOLD_MM} mm")
print(f"Accuracy: {dashboard_accuracy*100:.2f}%")
print(f"Precision: {dashboard_precision*100:.2f}%")
print(f"Recall: {dashboard_recall*100:.2f}%")
print(f"F1 score: {dashboard_f1:.3f}")
print(classification_report(actual_classes, predicted_classes,
                            labels=[0, 1], target_names=["No Rain", "Rain"],
                            zero_division=0))

cm = confusion_matrix(actual_classes, predicted_classes, labels=[0, 1])
cm_fig = go.Figure(data=go.Heatmap(
    z=cm, x=["Predicted No Rain", "Predicted Rain"],
    y=["Actual No Rain", "Actual Rain"], text=cm,
    texttemplate="%{text}", colorscale="Blues"
))
cm_fig.update_layout(title="Threshold-Based Confusion Matrix",
                     template="plotly_white", height=380)
cm_fig.show()

Threshold: 0.1 mm
Accuracy: 79.73%
Precision: 71.86%
Recall: 94.24%
F1 score: 0.815
              precision    recall  f1-score   support

     No Rain       0.93      0.67      0.78       422
        Rain       0.72      0.94      0.82       382

    accuracy                           0.80       804
   macro avg       0.82      0.80      0.80       804
weighted avg       0.83      0.80      0.79       804



## 10. Recursive seven-day backtest

In [10]:
all_predictions, all_actuals, all_baseline = [], [], []

# Non-overlapping 7-day windows in the held-out test period.
for start in range(split_index, len(data) - 6, 7):
    history_raw = data[start-7:start].copy()
    current_sequence = feature_scaler.transform(history_raw)
    week_predictions = []

    for day in range(7):
        model_input = current_sequence.reshape(1, 7, 5)
        pred_scaled_day = model.predict(model_input, verbose=0)
        pred_mm = float(target_scaler.inverse_transform(pred_scaled_day)[0, 0])
        pred_mm = max(0.0, pred_mm)
        week_predictions.append(pred_mm)

        # Recursively update rainfall; hold other weather features at last observed values.
        next_raw = feature_scaler.inverse_transform(
            current_sequence[-1].reshape(1, 5)
        )[0]
        next_raw[4] = pred_mm
        next_scaled = feature_scaler.transform(next_raw.reshape(1, 5))[0]
        current_sequence = np.vstack([current_sequence[1:], next_scaled])

    all_predictions.extend(week_predictions)
    all_actuals.extend(data[start:start+7, 4])
    all_baseline.extend(np.repeat(history_raw[-1, 4], 7))

all_predictions = np.asarray(all_predictions, dtype=float)
all_actuals = np.asarray(all_actuals, dtype=float)
all_baseline = np.asarray(all_baseline, dtype=float)

forecast_mae = mean_absolute_error(all_actuals, all_predictions)
forecast_rmse = np.sqrt(mean_squared_error(all_actuals, all_predictions))
baseline_mae = mean_absolute_error(all_actuals, all_baseline)
baseline_rmse = np.sqrt(mean_squared_error(all_actuals, all_baseline))

print("SEVEN-DAY RECURSIVE BACKTEST")
print("Forecast samples:", len(all_actuals))
print(f"LSTM MAE: {forecast_mae:.3f} mm")
print(f"LSTM RMSE: {forecast_rmse:.3f} mm")
print(f"Persistence baseline MAE: {baseline_mae:.3f} mm")
print(f"Persistence baseline RMSE: {baseline_rmse:.3f} mm")

horizon_rows = []
for day in range(7):
    a, p = all_actuals[day::7], all_predictions[day::7]
    horizon_rows.append({
        "Forecast Day": f"Day {day+1}",
        "MAE (mm)": mean_absolute_error(a, p),
        "RMSE (mm)": np.sqrt(mean_squared_error(a, p))
    })
horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df.round(3))

hfig = go.Figure()
hfig.add_trace(go.Bar(x=horizon_df["Forecast Day"], y=horizon_df["MAE (mm)"],
                      text=horizon_df["MAE (mm)"].map(lambda v: f"{v:.2f}"),
                      textposition="outside"))
hfig.update_layout(title="MAE by Forecast Day", xaxis_title="Forecast horizon",
                   yaxis_title="MAE (mm)", template="plotly_white", height=400)
hfig.show()

rfig = go.Figure()
rfig.add_trace(go.Scatter(y=all_actuals[:150], mode="lines", name="Actual rainfall"))
rfig.add_trace(go.Scatter(y=all_predictions[:150], mode="lines", name="Recursive prediction"))
rfig.update_layout(title="Recursive 7-Day Backtest: Actual vs Predicted",
                   xaxis_title="Forecast observation", yaxis_title="Rainfall (mm)",
                   template="plotly_white", height=420)
rfig.show()

SEVEN-DAY RECURSIVE BACKTEST
Forecast samples: 798
LSTM MAE: 2.703 mm
LSTM RMSE: 5.945 mm
Persistence baseline MAE: 3.902 mm
Persistence baseline RMSE: 8.090 mm


,Forecast Day,MAE (mm),RMSE (mm)
0,Day 1,2.481,4.955
1,Day 2,2.173,4.403
2,Day 3,2.616,7.718
3,Day 4,2.756,5.844
4,Day 5,2.211,4.085
5,Day 6,3.060,6.427
6,Day 7,3.625,7.193


## 11. Save model, scalers, dataset, and evaluation outputs

In [11]:
ARTIFACT_DIR = Path("rainvision_ai_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

model.save(ARTIFACT_DIR / "rainfall_prediction_model.keras")
joblib.dump(feature_scaler, ARTIFACT_DIR / "feature_scaler.pkl")
joblib.dump(target_scaler, ARTIFACT_DIR / "target_scaler.pkl")
df.to_csv(ARTIFACT_DIR / "hyderabad_rainfall_dataset.csv", index=False)
horizon_df.to_csv(ARTIFACT_DIR / "seven_day_forecast_performance.csv", index=False)

pd.DataFrame({
    "Actual_Rainfall": all_actuals,
    "Predicted_Rainfall": all_predictions,
    "Baseline_Rainfall": all_baseline
}).to_csv(ARTIFACT_DIR / "seven_day_forecast_results.csv", index=False)

pd.DataFrame([{
    "LSTM_MAE_mm": one_day_mae,
    "LSTM_RMSE_mm": one_day_rmse,
    "Zero_Baseline_MAE_mm": zero_mae,
    "Zero_Baseline_RMSE_mm": zero_rmse,
    "Rain_Threshold_mm": RAIN_THRESHOLD_MM,
    "Threshold_Accuracy": dashboard_accuracy,
    "Threshold_Precision": dashboard_precision,
    "Threshold_Recall": dashboard_recall,
    "Threshold_F1": dashboard_f1
}]).to_csv(ARTIFACT_DIR / "one_day_model_metrics.csv", index=False)

pd.DataFrame([{
    "LSTM_7day_MAE_mm": forecast_mae,
    "LSTM_7day_RMSE_mm": forecast_rmse,
    "Persistence_Baseline_MAE_mm": baseline_mae,
    "Persistence_Baseline_RMSE_mm": baseline_rmse,
    "Forecast_Samples": len(all_actuals)
}]).to_csv(ARTIFACT_DIR / "seven_day_summary_metrics.csv", index=False)

print("Saved artifacts:")
for item in sorted(ARTIFACT_DIR.iterdir()):
    print(item.name, f"({item.stat().st_size/1024:.1f} KB)")

Saved artifacts:
feature_scaler.pkl (0.9 KB)
hyderabad_rainfall_dataset.csv (137.2 KB)
one_day_model_metrics.csv (0.3 KB)
rainfall_prediction_model.keras (400.7 KB)
seven_day_forecast_performance.csv (0.3 KB)
seven_day_forecast_results.csv (18.2 KB)
seven_day_summary_metrics.csv (0.2 KB)
target_scaler.pkl (0.7 KB)


## 12. Build and launch the blue-and-white Gradio dashboard

In [12]:
latest_weather = df[features].tail(7).reset_index(drop=True).copy()

def create_history_chart():
    recent = df.tail(30)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=recent["Date"], y=recent["Rainfall"],
                         marker_color="#4285F4", name="Historical rainfall"))
    fig.update_layout(title="Rainfall History — Last 30 Days",
                      xaxis_title="Date", yaxis_title="Rainfall (mm)",
                      template="plotly_white", height=360)
    return fig

def create_comparison_chart():
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=actual_rainfall[:150], mode="lines",
                             name="Actual", line=dict(color="#2563EB", width=2)))
    fig.add_trace(go.Scatter(y=predicted_rainfall[:150], mode="lines",
                             name="LSTM prediction", line=dict(color="#F59E0B", width=2)))
    fig.update_layout(title="Actual vs Predicted Rainfall — Test Set",
                      xaxis_title="Test observations", yaxis_title="Rainfall (mm)",
                      template="plotly_white", height=360)
    return fig

def create_confusion_chart():
    matrix = confusion_matrix(actual_classes, predicted_classes, labels=[0, 1])
    fig = go.Figure(data=go.Heatmap(
        z=matrix, x=["No Rain", "Rain"], y=["No Rain", "Rain"],
        text=matrix, texttemplate="%{text}", colorscale="Blues"
    ))
    fig.update_layout(title="Threshold-Based Confusion Matrix",
                      xaxis_title="Predicted class", yaxis_title="Actual class",
                      template="plotly_white", height=360)
    return fig

def generate_prediction(weather_input):
    if isinstance(weather_input, pd.DataFrame):
        weather_input = weather_input[features].values
    weather_input = np.asarray(weather_input, dtype=float)

    if weather_input.shape != (7, 5):
        raise gr.Error("Please enter exactly 7 rows and 5 columns.")
    if not np.isfinite(weather_input).all():
        raise gr.Error("Please enter valid numerical values.")
    if np.any((weather_input[:, 1] < 0) | (weather_input[:, 1] > 100)):
        raise gr.Error("Humidity must be between 0 and 100.")
    if np.any(weather_input[:, 4] < 0):
        raise gr.Error("Rainfall cannot be negative.")

    sequence = feature_scaler.transform(weather_input)
    future_values = []

    for _ in range(7):
        pred_scaled = model.predict(sequence.reshape(1, 7, 5), verbose=0)
        pred_mm = max(0.0, float(target_scaler.inverse_transform(pred_scaled)[0, 0]))
        future_values.append(pred_mm)

        # The model predicts rainfall only. Hold other weather features at
        # their most recently observed values for this recursive demonstration.
        next_raw = feature_scaler.inverse_transform(sequence[-1].reshape(1, 5))[0]
        next_raw[4] = pred_mm
        next_scaled = feature_scaler.transform(next_raw.reshape(1, 5))[0]
        sequence = np.vstack([sequence[1:], next_scaled])

    labels = [f"Day {i}" for i in range(1, 8)]
    table = pd.DataFrame({
        "Forecast Day": labels,
        "Predicted Rainfall (mm)": [round(v, 2) for v in future_values]
    })

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=labels, y=future_values, marker_color="#2563EB",
        text=[f"{v:.2f} mm" for v in future_values],
        textposition="outside", cliponaxis=False,
        name="Predicted rainfall"
    ))
    fig.update_layout(
        title="Experimental Recursive 7-Day Rainfall Estimate",
        xaxis_title="Forecast day", yaxis_title="Rainfall (mm)",
        template="plotly_white", height=420, showlegend=False,
        yaxis=dict(range=[0, max(1.0, max(future_values) * 1.35)])
    )

    result = (
        "## 🌧️ Seven-Day Rainfall Estimate\n\n"
        "**Location:** Hyderabad, Telangana  \n"
        "**Model:** LSTM deep learning  \n\n"
        + table.to_markdown(index=False)
        + "\n\n*Experimental recursive estimates, not a verified live weather forecast. "
          "Future non-rainfall weather variables are held at their last observed values.*"
    )
    return result, round(future_values[0], 2), fig

css = """
.gradio-container {max-width: 1450px !important; background: #f4f7fc !important;}
#header {background: linear-gradient(120deg,#102b55,#2463a5); color:white;
padding:26px; border-radius:16px; margin-bottom:16px;}
#header h1, #header p {color:white !important;}
.metric-card {background:white; padding:18px; border-radius:13px;
border:1px solid #e3eaf4; min-height:100px;}
.metric-label {color:#64748b; font-size:13px;}
.metric-value {color:#173d70; font-size:25px; font-weight:700;}
#predict-btn {background:#1769aa !important; color:white !important;
font-weight:bold !important; min-height:48px;}
footer {display:none !important;}
"""

with gr.Blocks(title="RainVision AI — Rainfall Analytics", css=css) as demo:
    gr.HTML("""
    <div id="header">
      <h1>🌧️ RainVision AI</h1>
      <p>Intelligent Rainfall Prediction & Weather Analytics</p>
      <p>HYDERABAD, TELANGANA | LSTM DEEP LEARNING SYSTEM</p>
    </div>
    """)

    with gr.Row():
        overview_btn = gr.Button("Overview")
        prediction_btn = gr.Button("AI Prediction")
        analytics_btn = gr.Button("Analytics")
        performance_btn = gr.Button("Model Performance")

    with gr.Column(visible=True) as overview_section:
        gr.Markdown("## Weather Analytics Overview")
        with gr.Row():
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Average Rainfall</div><div class="metric-value">{df["Rainfall"].mean():.2f} mm</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Average Temperature</div><div class="metric-value">{df["Temperature"].mean():.1f} °C</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Average Humidity</div><div class="metric-value">{df["Humidity"].mean():.1f} %</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Average Wind Speed</div><div class="metric-value">{df["WindSpeed"].mean():.1f}</div></div>')
        gr.Markdown("### Historical Rainfall Analytics")
        gr.Plot(value=create_history_chart())

    with gr.Column(visible=False) as prediction_section:
        gr.Markdown("## AI Rainfall Prediction")
        gr.Markdown("Edit the seven historical weather observations, then generate a recursive seven-day estimate.")
        weather_table = gr.Dataframe(
            value=latest_weather, headers=features, datatype="number",
            row_count=7, col_count=(5, "fixed"),
            interactive=True, label="Seven-Day Weather Input"
        )
        predict_btn = gr.Button("🌧️ Generate 7-Day AI Prediction", elem_id="predict-btn")
        with gr.Row():
            prediction_result = gr.Markdown("Your forecast will appear here.")
            prediction_value = gr.Number(label="Day 1 Predicted Rainfall (mm)",
                                         precision=2, interactive=False)
        prediction_chart = gr.Plot()
        predict_btn.click(
            fn=generate_prediction, inputs=weather_table,
            outputs=[prediction_result, prediction_value, prediction_chart]
        )

    with gr.Column(visible=False) as analytics_section:
        gr.Markdown("## Rainfall Analytics")
        gr.Plot(value=create_history_chart())
        gr.Plot(value=create_comparison_chart())

    with gr.Column(visible=False) as performance_section:
        gr.Markdown("## LSTM Model Performance")
        with gr.Row():
            gr.HTML(f'<div class="metric-card"><div class="metric-label">One-Day MAE</div><div class="metric-value">{one_day_mae:.3f} mm</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">One-Day RMSE</div><div class="metric-value">{one_day_rmse:.3f} mm</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Threshold Accuracy</div><div class="metric-value">{dashboard_accuracy*100:.2f}%</div></div>')
            gr.HTML(f'<div class="metric-card"><div class="metric-label">Threshold F1</div><div class="metric-value">{dashboard_f1:.3f}</div></div>')
        gr.Markdown("### Seven-Day Recursive Backtest")
        gr.Markdown(
            f"- LSTM MAE: **{forecast_mae:.3f} mm**\n"
            f"- LSTM RMSE: **{forecast_rmse:.3f} mm**\n"
            f"- Persistence baseline MAE: **{baseline_mae:.3f} mm**\n"
            f"- Persistence baseline RMSE: **{baseline_rmse:.3f} mm**"
        )
        gr.Markdown(
            "### Threshold-Based Classification Metrics\n"
            f"- Precision: **{dashboard_precision*100:.2f}%**\n"
            f"- Recall: **{dashboard_recall*100:.2f}%**\n"
            f"- F1 score: **{dashboard_f1:.3f}**\n\n"
            "*These classification metrics are obtained by thresholding regression predictions.*"
        )
        gr.Plot(value=create_confusion_chart())

    def navigate(page):
        return (
            gr.update(visible=page == "overview"),
            gr.update(visible=page == "prediction"),
            gr.update(visible=page == "analytics"),
            gr.update(visible=page == "performance")
        )

    overview_btn.click(lambda: navigate("overview"), outputs=[
        overview_section, prediction_section, analytics_section, performance_section
    ])
    prediction_btn.click(lambda: navigate("prediction"), outputs=[
        overview_section, prediction_section, analytics_section, performance_section
    ])
    analytics_btn.click(lambda: navigate("analytics"), outputs=[
        overview_section, prediction_section, analytics_section, performance_section
    ])
    performance_btn.click(lambda: navigate("performance"), outputs=[
        overview_section, prediction_section, analytics_section, performance_section
    ])

    gr.Markdown("""
    ---
    **RainVision AI | Academic Deep Learning Project**
    Python • TensorFlow • LSTM • Pandas • Plotly • Gradio
    Historical-data-based experimental predictions.
    """)

print("Dashboard built. Launching...")
demo.launch(share=True, debug=False, theme=gr.themes.Soft())

Dashboard built. Launching...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b77e9688853fdddb7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 13. Download the complete project artifacts

In [13]:
# Save all generated artifacts in one ZIP file.
zip_path = "rainvision_ai_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in sorted(ARTIFACT_DIR.iterdir()):
        zf.write(item, arcname=item.name)

print("Created:", zip_path)
with zipfile.ZipFile(zip_path, "r") as zf:
    print("Included files:")
    for name in zf.namelist():
        print("-", name)

# Run this separately when you want the ZIP downloaded to your computer:
# from google.colab import files
# files.download("rainvision_ai_artifacts.zip")

Created: rainvision_ai_artifacts.zip
Included files:
- feature_scaler.pkl
- hyderabad_rainfall_dataset.csv
- one_day_model_metrics.csv
- rainfall_prediction_model.keras
- seven_day_forecast_performance.csv
- seven_day_forecast_results.csv
- seven_day_summary_metrics.csv
- target_scaler.pkl


## 14. Download the ZIP to your computer

After the dashboard has launched and you have finished testing it, run:

```python
from google.colab import files
files.download("rainvision_ai_artifacts.zip")
```

### GitHub submission notes
- Upload this notebook and the generated artifact files to your repository.
- If the `.keras` file is too large for GitHub's standard upload limit, use Git LFS or link to a suitable external file host.
- Do not upload passwords, API keys, or private credentials.
- A fresh run can produce different metrics if the source archive or training run changes. Use the results produced by the notebook run you submit.